In [0]:
from pyspark.sql.functions import col, when, abs as spark_abs, round

# Rebuild Silver
df_silver = (
    spark.table("fraud_bronze")
    .drop("_rescued_data")
    .withColumn("Amount", col("Amount").cast("double"))
    .withColumn("Class", col("Class").cast("integer"))
    .filter(col("Amount") >= 0)
    .filter(col("Class").isin(0, 1))
    .dropDuplicates()
    .na.drop(subset=["Amount", "Class"])
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("fraud_silver")
print(f"Silver rows: {df_silver.count()}")
print(f"Silver columns: {df_silver.columns}")

# Rebuild Gold
df_gold = (
    spark.table("fraud_silver")
    .withColumn("Amount_log", round(spark_abs(col("Amount") + 1), 6))
    .withColumn("Amount_scaled", round(col("Amount") / 25691.16, 6))
    .withColumn("high_value", when(col("Amount") > 1000, 1).otherwise(0))
    .drop("Time")
)

df_gold.write.format("delta").mode("overwrite").saveAsTable("fraud_gold")
print(f"Gold rows: {df_gold.count()}")
print(f"Gold columns: {df_gold.columns}")

Silver rows: 283726
Silver columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
Gold rows: 283726
Gold columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class', '_rescued_data', 'Amount_log', 'Amount_scaled', 'high_value']


In [0]:
spark.sql("DROP TABLE IF EXISTS fraud_gold")
spark.sql("DROP TABLE IF EXISTS fraud_silver")

from pyspark.sql.functions import col, when, abs as spark_abs, round

# Rebuild Silver explicitly dropping _rescued_data
df_silver = (
    spark.table("fraud_bronze")
    .drop("_rescued_data")
    .withColumn("Amount", col("Amount").cast("double"))
    .withColumn("Class", col("Class").cast("integer"))
    .filter(col("Amount") >= 0)
    .filter(col("Class").isin(0, 1))
    .dropDuplicates()
    .na.drop(subset=["Amount", "Class"])
)

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_silver")

print(f"Silver columns: {spark.table('fraud_silver').columns}")

# Rebuild Gold
df_gold = (
    spark.table("fraud_silver")
    .withColumn("Amount_log", round(spark_abs(col("Amount") + 1), 6))
    .withColumn("Amount_scaled", round(col("Amount") / 25691.16, 6))
    .withColumn("high_value", when(col("Amount") > 1000, 1).otherwise(0))
    .drop("Time")
)

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_gold")

print(f"Gold columns: {spark.table('fraud_gold').columns}")

Silver columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
Gold columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class', 'Amount_log', 'Amount_scaled', 'high_value']
